In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("../utils"))
sys.path.insert(0, os.path.abspath("../sdlarch-rl"))


from sdlarch_rl.utils.utils import get_last_index, GenericCNN, TimeLimit, FrameSkip
from sdlarch_rl import make
from stable_baselines3.common.atari_wrappers import WarpFrame
import time
import cv2
import numpy as np
import keyboard
from stable_baselines3.common.atari_wrappers import WarpFrame
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv
from pathlib import Path
import pygame
from inputs import get_gamepad, devices
import threading
from imitation.data.types import Trajectory
import torch as th
import os
import re
from utils import get_last_index, LSTMWrapper
import gymnasium as gym
from final_fight import FinalFightActionWrapper, make_env
from stable_baselines3.common.policies import ActorCriticPolicy
import sounddevice as sd

env = make_env(random=False, skip=False)()

arate = env.unwrapped.em.get_audio_rate()

# --- CONFIGS ---
SCALE = 1/4
SCREEN_WIDTH = 854 # int(640 * SCALE)
SCREEN_HEIGHT = 480 # int(480 * SCALE)
NUMBER_OF_ACTIONS = 6

# pygame.mixer.init(frequency=int(arate), size=-16, channels=2)
# channel = pygame.mixer.Channel(0)
audio_stream = sd.OutputStream(samplerate=int(arate), channels=2, dtype='int16')
audio_stream.start()

clock = pygame.time.Clock()

class RendererThread(threading.Thread):
    def __init__(self):
        super().__init__(daemon=True)
        self.img = None
        self.color = (0, 0, 255)
        self.count_record = 0
        self.running = True
        self.lock = threading.Lock()
        self.fps = 0

    def update_data(self, img):
        with self.lock:
            self.img = img

    def run(self):
        pygame.init()
        window = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT), pygame.HWSURFACE | pygame.DOUBLEBUF)
        pygame.display.set_caption("Final Fight Capture")
        font = pygame.font.SysFont("Arial", 24)
        
        while self.running:
            pygame.event.pump()
            
            img_to_render = None
            with self.lock:
                if self.img is not None:
                    img_to_render = self.img


            if img_to_render is not None:
                img_rgb = cv2.cvtColor(img_to_render, cv2.COLOR_BGR2RGB)
                surface = pygame.image.frombuffer(img_rgb.flatten(), (SCREEN_WIDTH, SCREEN_HEIGHT), 'RGB')
                
                window.blit(surface, (0, 0))

            
                pygame.display.flip()
            
            time.sleep(1/30)



gamepad = devices.gamepads[0]
STATE = {k: 0 for k in ["UP", "DOWN", "LEFT", "RIGHT", "A", "B", "X", "START", "CAM_X", "CAM_Y", "LT", "RT", "L3"]}
lock = threading.Lock()

DEADZONE = 10000
NORM = 32768

def input_loop():
    while True:
        for e in gamepad.read():
            with lock:
                if e.code == "ABS_HAT0X":
                    STATE["LEFT"]  = 1 if e.state == -1 else 0
                    STATE["RIGHT"] = 1 if e.state == 1 else 0

                elif e.code == "ABS_HAT0Y":
                    STATE["UP"]   = 1 if e.state == -1 else 0
                    STATE["DOWN"] = 1 if e.state == 1 else 0

                elif e.code == "ABS_X":
                    if e.state > DEADZONE:
                        STATE["LEFT"] = 0
                        STATE["RIGHT"] = 1
                    elif e.state < -DEADZONE:
                        STATE["LEFT"] = 1
                        STATE["RIGHT"] = 0
                    else:
                        STATE["LEFT"] = 0
                        STATE["RIGHT"] = 0

                elif e.code == "ABS_Y":
                    if e.state > DEADZONE:
                        STATE["UP"] = 1
                        STATE["DOWN"] = 0
                    elif e.state < -DEADZONE:
                        STATE["UP"] = 0
                        STATE["DOWN"] = 1
                    else:
                        STATE["UP"] = 0
                        STATE["DOWN"] = 0

                # RIGHT STICK X
                elif e.code == "ABS_RX":
                    if abs(e.state) > DEADZONE:
                        STATE["CAM_X"] = e.state / NORM
                    else:
                        STATE["CAM_X"] = 0

                # RIGHT STICK Y
                elif e.code == "ABS_RY":
                    if abs(e.state) > DEADZONE:
                        STATE["CAM_Y"] = e.state / NORM
                    else:
                        STATE["CAM_Y"] = 0

                elif e.code == "BTN_SOUTH":
                    STATE["A"] = e.state

                elif e.code == "BTN_EAST":
                    STATE["B"] = e.state

                elif e.code == "BTN_WEST":
                    STATE["X"] = e.state

                elif e.code == "BTN_START" or e.code == "BTN_MENU" \
                    or e.code == "BTN_SELECT" or e.code == "BTN_MODE"  \
                    or e.code == "BTN_OPTIONS":
                    STATE["START"] = e.state

                elif e.code == "BTN_THUMBL":
                    STATE["L3"] = e.state

                elif e.code == "ABS_Z":
                    value = e.state / 255.0
                    if value < 0.5:
                        value = 0
                    STATE["LT"] = value
                
                elif e.code == "ABS_RZ":
                    value = e.state / 255.0
                    if value < 0.5:
                        value = 0
                    STATE["RT"] = value

t_input = threading.Thread(target=input_loop, daemon=True)
t_input.start()

# Start render thread
renderer = RendererThread()
renderer.start()


print("Done! Press 'K' to start/stop the record.")

fps_avg_frame_count = 0
fps_start_time = time.time()
actual_fps = 0

obs = env.reset()
is_paused = False


while True:
    loop_start = time.time()

    action = np.zeros(NUMBER_OF_ACTIONS, dtype=np.float32)

    with lock:
        received_action = [
            STATE["UP"], # 0 
            STATE["DOWN"], 
            STATE["LEFT"], 
            STATE["RIGHT"],
            STATE["A"], 
            STATE["B"], 
            STATE["X"],
            STATE["LT"], # 7
            STATE["RT"],
            STATE["L3"],
            STATE["CAM_X"], # 10
            STATE["CAM_Y"], # 11
            STATE["START"],
        ]

        if keyboard.is_pressed('up') or received_action[0]: action[0] = 1
        if keyboard.is_pressed('down') or received_action[1]: action[1] = 1
        if keyboard.is_pressed('left') or received_action[2]: action[2] = 1
        if keyboard.is_pressed('right') or received_action[3]: action[3] = 1
        if keyboard.is_pressed('i') or received_action[4]: action[4] = 1
        if received_action[6]: action[5] = 1

        if keyboard.is_pressed('r'):
            env.reset()
            time.sleep(0.3)

        if keyboard.is_pressed('p'):
            is_paused = not is_paused
            time.sleep(0.3)

            

    if is_paused:
        continue


    obs, rew, done, trunc, info = env.step(action)
    sound = env.em.get_audio()

    if sound is not None:
        audio_stream.write(sound.astype(np.int16))

    
    img = env.render()
    if obs is not None:
        # img = obs[0, :, :, -1]
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_resized =  cv2.resize(img, (SCREEN_WIDTH, SCREEN_HEIGHT), interpolation=cv2.INTER_NEAREST)
        renderer.update_data(img_resized)


    clock.tick(60)

renderer.running = False
pygame.quit()

D:\anaconda3\envs\env311\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


statename is None setting to default state
Done! Press 'K' to start/stop the record.


D:\anaconda3\envs\env311\Lib\site-packages\gymnasium\core.py:311: UserWarning: WARN: env.em to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.em` for environment variables or `env.get_wrapper_attr('em')` that will search the reminding wrappers.
  logger.warn(
